# 04 — 3D Segmentation Inference

## Objective

This notebook applies the trained 2D U-Net to complete T2W MRI volumes.

Each axial slice is preprocessed and resized to the network input size,
segmented independently, and then restored to the original in-plane
resolution.

The resulting binary slices are reassembled into a 3D prostate mask
with the same array dimensions as the original MRI volume.

## 1. Environment Setup

The project root is added to the Python path so that the notebook can
access the reusable modules implemented in the `src` package.

In [8]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II


## 2. Imports and Paths

This section imports the modules required to load an original PI-CAI
T2W volume, restore the trained U-Net checkpoint, and run full-volume
segmentation inference.

In [9]:
import torch
import numpy as np
import pandas as pd
import importlib
import SimpleITK as sitk

In [10]:
import src.data.paths
importlib.reload(src.data.paths)

import src.data.mapping
importlib.reload(src.data.mapping)

import src.data.loader
importlib.reload(src.data.loader)

import src.segmentation.model
importlib.reload(src.segmentation.model)

import src.segmentation.inference
importlib.reload(src.segmentation.inference)

<module 'src.segmentation.inference' from 'c:\\Users\\Luiz Trindade\\Documents\\penguin-adventure\\TCC_II\\src\\segmentation\\inference.py'>

In [11]:
from src.data.paths import (
    CHECKPOINTS_DIR,
)

from src.data.mapping import (
    get_case_image_paths,
    find_whole_gland_mask,
)

from src.data.loader import (
    load_medical_image,
)

from src.segmentation.model import (
    UNet,
)

from src.segmentation.inference import (
    predict_volume_masks,
    save_predicted_mask,
    predict_and_save_cases,
)

In [12]:
checkpoint_path = (
    CHECKPOINTS_DIR
    / "unet_whole_gland_best.pth"
)

patient_id = "10001"
study_id = "1000001"

image_paths = get_case_image_paths(
    patient_id,
    study_id,
)

t2w_path = image_paths["t2w"]

print("T2W:", t2w_path)
print("Checkpoint:", checkpoint_path)

T2W: C:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\src\data\raw\picai\10001\10001_1000001_t2w.mha
Checkpoint: C:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\results\checkpoints\unet_whole_gland_best.pth


## 3. Original T2W Volume

The original T2W MRI volume is loaded before segmentation inference.

The original array dimensions are preserved so that the predicted
whole-gland mask can later be reconstructed in the same image space.

In [13]:
t2w_image, t2w_volume = load_medical_image(
    t2w_path
)

print(
    "Volume shape:",
    t2w_volume.shape,
)

print(
    "Volume dtype:",
    t2w_volume.dtype,
)

print(
    "Image size:",
    t2w_image.GetSize(),
)

print(
    "Image spacing:",
    t2w_image.GetSpacing(),
)

Volume shape: (21, 640, 640)
Volume dtype: uint16
Image size: (640, 640, 21)
Image spacing: (0.30000001192093, 0.30000001192093, 3.6000000778884846)


## 4. Model Loading

The U-Net architecture is initialized and the best checkpoint selected
during validation is restored.

The model is placed in evaluation mode before segmentation inference.

In [14]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = UNet(
    in_channels=1,
    out_channels=1,
).to(device)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Device:", device)
print(
    "Checkpoint epoch:",
    checkpoint["epoch"],
)

Device: cuda
Checkpoint epoch: 4


## 5. Full-Volume Segmentation Inference

The trained 2D U-Net is applied independently to every axial slice of
the original T2W volume.

Each slice is normalized and resized to the network input resolution of
256 × 256 pixels. After inference, the binary prediction is resized back
to the original in-plane resolution using nearest-neighbor interpolation.

Unlike the training-data preparation procedure, all axial slices are
processed during inference. Ground-truth masks are therefore not used
to determine which slices should be segmented.

In [15]:
predicted_volume = predict_volume_masks(
    model=model,
    volume=t2w_volume,
    device=device,
    threshold=0.5,
)

print(
    "Original volume shape:",
    t2w_volume.shape,
)

print(
    "Predicted mask shape:",
    predicted_volume.shape,
)

print(
    "Predicted mask dtype:",
    predicted_volume.dtype,
)

print(
    "Predicted mask values:",
    sorted(set(predicted_volume.ravel())),
)

print(
    "Predicted positive voxels:",
    int(predicted_volume.sum()),
)

Original volume shape: (21, 640, 640)
Predicted mask shape: (21, 640, 640)
Predicted mask dtype: uint8
Predicted mask values: [np.uint8(0), np.uint8(1)]
Predicted positive voxels: 284225


## 6. 3D Segmentation Validation

For validation of the reconstruction procedure, the predicted 3D mask
is compared with the reference whole-gland annotation available for the
selected test case.

Dice and Intersection over Union (IoU) are calculated over the complete
3D volume rather than independently for each axial slice.

The reference annotation is used only to evaluate the segmentation and
does not participate in the inference procedure.

In [16]:
mask_path = find_whole_gland_mask(
    patient_id,
    study_id,
)

mask_image, ground_truth_volume = (
    load_medical_image(mask_path)
)

ground_truth_positive_slices = [
    index
    for index in range(
        ground_truth_volume.shape[0]
    )
    if ground_truth_volume[index].sum() > 0
]

print(
    "Ground-truth shape:",
    ground_truth_volume.shape,
)

print(
    "Ground-truth positive slices:",
    ground_truth_positive_slices,
)

print(
    "Number of ground-truth positive slices:",
    len(ground_truth_positive_slices),
)

Ground-truth shape: (21, 640, 640)
Ground-truth positive slices: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
Number of ground-truth positive slices: 17


In [17]:
ground_truth_binary = (
    ground_truth_volume > 0
).astype("uint8")

intersection = (
    predicted_volume
    * ground_truth_binary
).sum()

predicted_voxels = (
    predicted_volume.sum()
)

ground_truth_voxels = (
    ground_truth_binary.sum()
)

volume_dice = (
    2.0 * intersection
    / (
        predicted_voxels
        + ground_truth_voxels
    )
)

union = (
    (
        predicted_volume
        + ground_truth_binary
    ) > 0
).sum()

volume_iou = (
    intersection
    / union
)

print(
    "3D Dice:",
    float(volume_dice),
)

print(
    "3D IoU:",
    float(volume_iou),
)

print(
    "Predicted voxels:",
    int(predicted_voxels),
)

print(
    "Ground-truth voxels:",
    int(ground_truth_voxels),
)

3D Dice: 0.9402849921033587
3D IoU: 0.887299873170305
Predicted voxels: 284225
Ground-truth voxels: 276773


For the selected test case, the reconstructed segmentation achieved a
3D Dice coefficient of 0.9403 and an IoU of 0.8873.

The reference annotation contains prostate tissue across 17 axial
slices, while the predicted segmentation covers 16 slices. Despite this
difference at the boundary of the gland, the reconstructed volume shows
high spatial agreement with the reference annotation.

## 7. Physical-Space Reconstruction

The predicted NumPy volume contains the segmentation in the original
array dimensions, but it does not independently store medical-image
spatial metadata.

A SimpleITK image is therefore created from the predicted volume and the
size, spacing, origin, and direction information of the original T2W MRI
is preserved.

This ensures that the predicted segmentation occupies the same physical
image space as the source MRI.

In [18]:
predicted_mask_image = sitk.GetImageFromArray(
    predicted_volume.astype("uint8")
)

predicted_mask_image.CopyInformation(
    t2w_image
)

print(
    "Predicted mask size:",
    predicted_mask_image.GetSize(),
)

print(
    "Predicted mask spacing:",
    predicted_mask_image.GetSpacing(),
)

print(
    "Predicted mask origin:",
    predicted_mask_image.GetOrigin(),
)

print(
    "Predicted mask direction:",
    predicted_mask_image.GetDirection(),
)

Predicted mask size: (640, 640, 21)
Predicted mask spacing: (0.30000001192093, 0.30000001192093, 3.6000000778884846)
Predicted mask origin: (-119.80861242576, -65.872387498812, -38.779758120899)
Predicted mask direction: (0.9996148108062515, -1.6670999975453016e-11, -0.02775301815627119, 0.00043710598429775216, 0.999875963366245, 0.01574378673580096, 0.027749575765060076, -0.015749853409600077, 0.9994908219500744)


## 8. Saving the Predicted 3D Mask

The reconstructed whole-gland segmentation is saved as an MHA medical
image.

The saved mask preserves the spatial metadata of the original T2W MRI,
allowing it to be used together with the source image in subsequent
radiomic feature extraction.

In [19]:
prediction_dir = (
    PROJECT_ROOT
    / "results"
    / "segmentation"
    / "predicted_masks"
)

prediction_dir.mkdir(
    parents=True,
    exist_ok=True,
)

predicted_mask_path = (
    prediction_dir
    / f"{patient_id}_{study_id}_whole_gland_pred.mha"
)

sitk.WriteImage(
    predicted_mask_image,
    str(predicted_mask_path),
)

print(
    "Predicted mask saved to:",
    predicted_mask_path,
)

Predicted mask saved to: c:\Users\Luiz Trindade\Documents\penguin-adventure\TCC_II\results\segmentation\predicted_masks\10001_1000001_whole_gland_pred.mha


### Saved Mask Verification

The saved segmentation is reloaded to verify that its binary content and
spatial metadata were preserved during serialization.

In [20]:
reloaded_mask_image, reloaded_mask_volume = (
    load_medical_image(
        predicted_mask_path
    )
)

print(
    "Reloaded shape:",
    reloaded_mask_volume.shape,
)

print(
    "Reloaded dtype:",
    reloaded_mask_volume.dtype,
)

print(
    "Reloaded values:",
    sorted(
        set(
            reloaded_mask_volume.ravel()
        )
    ),
)

print(
    "Same size:",
    reloaded_mask_image.GetSize()
    == t2w_image.GetSize(),
)

print(
    "Same spacing:",
    reloaded_mask_image.GetSpacing()
    == t2w_image.GetSpacing(),
)

print(
    "Same origin:",
    reloaded_mask_image.GetOrigin()
    == t2w_image.GetOrigin(),
)

print(
    "Same direction:",
    reloaded_mask_image.GetDirection()
    == t2w_image.GetDirection(),
)

Reloaded shape: (21, 640, 640)
Reloaded dtype: uint8
Reloaded values: [np.uint8(0), np.uint8(1)]
Same size: True
Same spacing: True
Same origin: True
Same direction: True


## 9. Inference Summary

The complete 3D segmentation inference workflow was successfully
validated on a test case.

The trained 2D U-Net was applied to every axial slice of the original
T2W MRI without using the reference mask for slice selection. The
predicted slices were restored to the original in-plane resolution and
reassembled into a binary 3D whole-gland segmentation.

For the evaluated case, the reconstructed segmentation achieved:

- 3D Dice coefficient: 0.9403
- 3D IoU: 0.8873

The predicted mask was subsequently converted to a SimpleITK image and
assigned the spatial metadata of the original T2W MRI. After saving and
reloading the mask, its size, spacing, origin, direction, and binary
representation were successfully preserved.

The resulting 3D whole-gland segmentation is therefore prepared for the
subsequent radiomic feature extraction stage.

# extra

In [21]:
from src.segmentation.inference import (
    predict_and_save_cases,
)

batch_results = []
failed_cases = []

for index, row in all_cases.iterrows():
    case_dataframe = pd.DataFrame(
        [row]
    )

    patient_id = str(
        row["patient_id"]
    )

    study_id = str(
        row["study_id"]
    )

    case_id = (
        f"{patient_id}_{study_id}"
    )

    print(
        f"[{index + 1}/{len(all_cases)}] "
        f"Processing {case_id}"
    )

    try:
        case_results = predict_and_save_cases(
            model=model,
            cases=case_dataframe,
            device=device,
            output_dir=predicted_masks_dir,
            threshold=0.5,
            image_size=(256, 256),
        )

        batch_results.extend(
            case_results
        )

    except Exception as error:
        failed_cases.append(
            {
                "patient_id": patient_id,
                "study_id": study_id,
                "case_id": case_id,
                "error": str(error),
            }
        )

        print(
            f"Failed {case_id}: "
            f"{error}"
        )

[1/1500] Processing 11443_1001467
[2/1500] Processing 10119_1000119
[3/1500] Processing 10483_1000491
[4/1500] Processing 10559_1000571
[5/1500] Processing 11452_1001476
[6/1500] Processing 10900_1000916
[7/1500] Processing 10467_1000475
[8/1500] Processing 11389_1001413
[9/1500] Processing 10902_1000918
[10/1500] Processing 10835_1000851
[11/1500] Processing 11338_1001361
[12/1500] Processing 10016_1000016
[13/1500] Processing 11076_1001098
[14/1500] Processing 10217_1000221
[15/1500] Processing 11212_1001235
[16/1500] Processing 11120_1001143
[17/1500] Processing 10805_1000821
[18/1500] Processing 10087_1000087
[19/1500] Processing 11444_1001468
[20/1500] Processing 10856_1000872
[21/1500] Processing 10721_1000737
[22/1500] Processing 10777_1000793
[23/1500] Processing 10214_1000218
[24/1500] Processing 10982_1001001
[25/1500] Processing 10659_1000675
[26/1500] Processing 10287_1000293
[27/1500] Processing 10210_1000214
[28/1500] Processing 10193_1000196
[29/1500] Processing 10509_10

In [22]:
generated_masks = list(
    predicted_masks_dir.glob(
        "*_whole_gland_pred.mha"
    )
)

print(
    "Generated masks:",
    len(generated_masks),
)

print(
    "Failed cases:",
    len(failed_cases),
)

Generated masks: 1500
Failed cases: 0
